In [3]:
import sys
import subprocess

# Install required packages using subprocess for consistency across all environments (Colab, local, any Python environment)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "praat-parselmouth", "--break-system-packages"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "textgrid", "--break-system-packages"])

0

In [4]:
import shutil
import os

# This deletes an outdated local copy of the cache when needed
cache_path = '/tmp/zhuang_audio'
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("🗑️ Old audio cache cleared!")
else:
    print("Cache already empty.")

Cache already empty.


In [5]:
import pandas as pd
import textgrid
import ipywidgets as widgets
import librosa
import librosa.display
import os
# Import praat-parselmouth to use Praat's pitch tracking algorithm
# Needs installing once via subprocess above
import parselmouth
import matplotlib.pyplot as plt
import numpy as np
# UI that allows filtering, displaying sounds nicely
from IPython.display import display, clear_output, Audio

In [6]:
# Detect environment (Colab vs. local computer)
try:
    # If this works, we're in Colab
    from google.colab import drive  # type: ignore[reportMissingImports]
    IN_COLAB = True
except ModuleNotFoundError:
    # If it fails, we are running locally on a Mac/PC
    IN_COLAB = False

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local Machine'}")

Environment: Local Machine


In [7]:
# Setup the data paths based on the environment
ERRORS_PATH = None # Default to None in Colab
if IN_COLAB:
    print("Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)

    # Google Drive path
    PROJECT_PATH = '/content/drive/MyDrive/Zhuang'
    METADATA_PATH = os.path.join(PROJECT_PATH, 'FileInfoForRIPA.csv')
    DRIVE_AUDIO_PATH = os.path.join(PROJECT_PATH, 'Audio')

    # Let's copy the files to the fast disk once.
    FAST_LOCAL_AUDIO = '/tmp/zhuang_audio'

    if os.path.exists(DRIVE_AUDIO_PATH):
        if not os.path.exists(FAST_LOCAL_AUDIO):
            print(f"Copying {DRIVE_AUDIO_PATH} to fast local Colab storage...")
            try:
                import shutil
                shutil.copytree(DRIVE_AUDIO_PATH, FAST_LOCAL_AUDIO)
                print("✅ Copy complete. Audio will load instantly.")
            except Exception as e:
                print(f"⚠️ Copy failed ({e}). Falling back to slow Google Drive.")
                FAST_LOCAL_AUDIO = DRIVE_AUDIO_PATH

        AUDIO_PATH = FAST_LOCAL_AUDIO
    else:
        print(f"❌ Error: Cannot find Zhuang folder in Google Drive at {PROJECT_PATH}")
        AUDIO_PATH = None

else:
    # Point to the CSV on your Mac's internal drive
    METADATA_PATH = '/Users/jeremyperkins/Documents/UVicMADS/Spring Semester/CSC575/Project/FileInfoForRIPA.csv'

    # Point to the Audio files on your External Hard Drive
    AUDIO_PATH = '/Volumes/JeremyLaptopExtHD/Documents/Linguistics/Projects/12_DuanZhuang_Jeremy/0_0_201512_Fieldwork_Nanning/3_FilesForAnalysis'

    # Point to the folder containing audio with errors
    ERRORS_PATH = '/Volumes/JeremyLaptopExtHD/Documents/Linguistics/Projects/12_DuanZhuang_Jeremy/0_0_201512_Fieldwork_Nanning/3a_ErrorFiles'
    if not os.path.exists(AUDIO_PATH):
         print(f"Error: Cannot find local Audio folder at {AUDIO_PATH}")
         AUDIO_PATH = None

In [8]:
# Load metadata file
if os.path.exists(METADATA_PATH):
    df = pd.read_csv(METADATA_PATH)
    print("✅ Metadata loaded!")

    df['Dialect'] = df['Filename'].astype(str).str[:2]
    df['Speaker'] = df['Filename'].astype(str).str[:4]
    df['WordID']  = df['Filename'].astype(str).str.split('_').str[2]

    n_before = len(df)
    df = df[df['Exclude'].astype(str).str.strip().str.lower() != 'skipped'].reset_index(drop=True)
    print(f"ℹRemoved {n_before - len(df)} skipped entries. {len(df)} usable rows remain.")
else:
    print(f"Could not find 'FileInfoForRIPA.csv' at {METADATA_PATH}")

✅ Metadata loaded!
ℹRemoved 979 skipped entries. 13443 usable rows remain.


In [9]:
# Build Audio File Index
_audio_files = {} # Dictionary mapping: filename -> full_path

if AUDIO_PATH and os.path.exists(AUDIO_PATH):
    for f in os.listdir(AUDIO_PATH):
        if f.lower().endswith('.wav'):
            _audio_files[f] = os.path.join(AUDIO_PATH, f)

# Safely skipped in Colab since ERRORS_PATH is None
if ERRORS_PATH and os.path.exists(ERRORS_PATH):
    for f in os.listdir(ERRORS_PATH):
        if f.lower().endswith('.wav'):
            _audio_files[f] = os.path.join(ERRORS_PATH, f)

print(f"Audio index built: {len(_audio_files)} files found across both folders.")

Audio index built: 13406 files found across both folders.


In [10]:
# ── Speaker median f0 cache (computed lazily on first use per speaker) ────────
speaker_f0_cache = {}

# Find the median f0 value per speaker across all samples for that speaker
def get_speaker_median_f0(speaker_id, n_samples=30):
    """Return median voiced f0 (Hz) for a speaker, sampling up to n_samples files."""
    if speaker_id in speaker_f0_cache:
        return speaker_f0_cache[speaker_id]

    spk_files = df[df['Speaker'] == speaker_id]['Filename'].tolist()
    if not spk_files:
        return None

    import random
    sample = random.sample(spk_files, min(n_samples, len(spk_files)))

    all_f0 = []
    for fname in sample:
        raw = str(fname).strip()
        if raw.lower().endswith('.wav'):
            raw = raw[:-4]

        target_file = None

        # Search through both folders
        for f, full_path in _audio_files.items():
            f_lower = f.lower()
            raw_lower = raw.lower()
            if f_lower == f"{raw_lower}.wav" or (f_lower.startswith(f"{raw_lower}_") and f_lower.endswith('.wav')):
                target_file = f
                wav_path = full_path # Instantly grab the correct folder path!
                break

        if target_file:
            try:
                snd  = parselmouth.Sound(wav_path)
                vals = snd.to_pitch().selected_array['frequency']
                all_f0.extend(vals[vals > 0].tolist())
            except Exception:
                pass

    if not all_f0:
        return None

    median_f0 = float(np.median(all_f0))
    speaker_f0_cache[speaker_id] = median_f0
    return median_f0  # ← FIXED: This was missing in v2, critical for pitch normalization

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# BUILD WIDGETS WITH DROPDOWN MENUS
# ══════════════════════════════════════════════════════════════════════════════

_dd_layout = widgets.Layout(width='160px')

# Build a sorted dropdown including 'All'
def _dd(col, description, layout=_dd_layout, numeric_sort=False):
    vals = df[col].dropna().unique().tolist()
    if numeric_sort:
        def _sort_key(v):
            try: return (0, int(v))
            except (ValueError, TypeError): return (1, str(v))
        vals = sorted(vals, key=_sort_key)
    else:
        vals = sorted(vals)
    opts = ['All'] + vals
    return widgets.Dropdown(options=opts, value='All',
                            description=description, layout=layout)

# ══════════════════════════════════════════════════════════════════════════════
# LEFT COLUMN FILTERS  (also used by single-column "Plot Audio")
# ══════════════════════════════════════════════════════════════════════════════
tone_dropdown         = _dd('ReclassifiedTone',        'Tone:')
vowel_length_dropdown = _dd('ReclassifiedVowelLength',  'Vowel Len:')
coda_consonant_dropdown = _dd('ReclassifiedCoda',      'Coda:')
onset_dropdown        = _dd('Onset',                   'Onset:')
exclude_dropdown      = _dd('Exclude',                  'Exclude:')
dict_match_dropdown   = _dd('DictionaryMatch',          'Dict Match:')
dialect_dropdown      = _dd('Dialect',                  'Dialect:')
speaker_dropdown      = _dd('Speaker',                  'Speaker:')
word_id_dropdown      = _dd('WordID',                   'Word ID:',  numeric_sort=True)
ons_type_dropdown     = _dd('OnsType',                  'Ons Type:')

# ══════════════════════════════════════════════════════════════════════════════
# RIGHT COLUMN FILTERS  (used only for side-by-side Compare)
# ══════════════════════════════════════════════════════════════════════════════
tone_R         = _dd('ReclassifiedTone',        'Tone:')
vowel_length_R = _dd('ReclassifiedVowelLength',  'Vowel Len:')
coda_R         = _dd('ReclassifiedCoda',         'Coda:')
onset_R        = _dd('Onset',                    'Onset:')
exclude_R      = _dd('Exclude',                   'Exclude:')
dict_match_R   = _dd('DictionaryMatch',           'Dict Match:')
dialect_R      = _dd('Dialect',                   'Dialect:')
speaker_R      = _dd('Speaker',                   'Speaker:')
word_id_R      = _dd('WordID',                    'Word ID:',  numeric_sort=True)
ons_type_R     = _dd('OnsType',                   'Ons Type:')

# ══════════════════════════════════════════════════════════════════════════════
# GLOBAL OPTIONS
# ══════════════════════════════════════════════════════════════════════════════
n_input = widgets.BoundedIntText(
    value=10, min=1, max=50, step=1,
    description='Num Files:',
    layout=widgets.Layout(width='150px')
)
spectrogram_dropdown = widgets.Dropdown(
    options=['y', 'n'], value='y',
    description='Show Spec:',
    layout=widgets.Layout(width='150px')
)
random_checkbox = widgets.Checkbox(
    value=False, description='Random?',
    indent=False, layout=widgets.Layout(width='120px')
)
normalize_f0_checkbox = widgets.Checkbox(
    value=False, description='Norm f0?',
    indent=False, layout=widgets.Layout(width='120px')
)

show_tg_checkbox = widgets.Checkbox(
    value=False, description='Show TextGrid?',
    indent=False, layout=widgets.Layout(width='140px')
)

# ══════════════════════════════════════════════════════════════════════════════
# BUTTONS AND OUTPUT AREAS
# ══════════════════════════════════════════════════════════════════════════════
search_button  = widgets.Button(description="Plot Audio", button_style='success')
compare_button = widgets.Button(description="Compare",    button_style='info')

output_box   = widgets.Output()
output_left  = widgets.Output(layout=widgets.Layout(width='50%'))
output_right = widgets.Output(layout=widgets.Layout(width='50%'))

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

# Apply all filters to df
def apply_filters(tone, vowel_length, coda, onset, exclude_status,
                  dict_match, dialect, speaker, word_id, ons_type):
    fdf = df
    if tone         != 'All': fdf = fdf[fdf['ReclassifiedTone']       == tone]
    if vowel_length != 'All': fdf = fdf[fdf['ReclassifiedVowelLength'] == vowel_length]
    if coda         != 'All': fdf = fdf[fdf['ReclassifiedCoda']        == coda]
    if onset        != 'All': fdf = fdf[fdf['Onset']                  == onset]
    if exclude_status != 'All': fdf = fdf[fdf['Exclude']               == exclude_status]
    if dict_match   != 'All': fdf = fdf[fdf['DictionaryMatch']         == dict_match]
    if dialect      != 'All': fdf = fdf[fdf['Dialect']                 == dialect]
    if speaker      != 'All': fdf = fdf[fdf['Speaker']                 == speaker]
    if word_id      != 'All': fdf = fdf[fdf['WordID']                  == word_id]
    if ons_type     != 'All': fdf = fdf[fdf['OnsType']                 == ons_type]
    return fdf

# Make global file-level cache (dictionary) to speed up file retrieval
_render_cache = {}

# Render one audio file
def render_file(row, show_spectrogram, ax_wave, ax_spec=None, ax_tg=None, normalize_f0=False):
    """Draw waveform + f0 line (and optionally spectrogram) for one audio row."""
    raw_name = str(row['Filename']).strip()

    # Strip .wav if it is already there so we can append suffixes cleanly
    if raw_name.lower().endswith('.wav'):
        raw_name = raw_name[:-4]

    audio_path = None
    file_name = None

    # Search through both folders
    for f, full_path in _audio_files.items():
        f_lower = f.lower()

        raw_lower = raw_name.lower()

        if f_lower == f"{raw_lower}.wav" or (f_lower.startswith(f"{raw_lower}_") and f_lower.endswith('.wav')):
            file_name = f
            audio_path = full_path # Instantly grab the correct folder path!
            break

    # If no file matched, bail out safely
    if audio_path is None:
        print(f"❌ Missing audio file: {raw_name} (Checked all suffixes)")
        return None, None, None

    # If the file's already cached, we skip; otherwise, read & cache the file
    if audio_path not in _render_cache:
      snd = parselmouth.Sound(audio_path)
      # Find wav file & read it
      y = snd.values[0].astype(np.float32)
      sr = int(snd.sampling_frequency)
      # Run Praat's pitch-tracking algorithm
      pitch = snd.to_pitch()
      # STFT for spectrogram
      D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
      # Save to cache
      _render_cache[audio_path] = {
          'y': y, 'sr': sr,
          # Use .copy to leave the original as-is so plotting doesn't permanently alter cached data later
          'pitch_values': pitch.selected_array['frequency'].copy(),
          'pitch_times': pitch.xs(),
          'D': D,
      }

    # Get file from cache
    c = _render_cache[audio_path]
    y, sr = c['y'], c['sr']
    pitch_values = c['pitch_values'].copy()
    pitch_times, D = c['pitch_times'], c['D']

    # Spectrogram data
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)

    # Panel 1 – waveform
    librosa.display.waveshow(y, sr=sr, ax=ax_wave)
    ax_wave.set(
        title=(f"{file_name}  |  Tone: {row['ReclassifiedTone']}"
               f"  |  VL: {row['ReclassifiedVowelLength']}"),
        ylabel='Amplitude'
    )

    # F0 overlay as a line
    pitch_values[pitch_values == 0] = np.nan   # hide voiceless segments
    ax_pitch = ax_wave.twinx()

    if normalize_f0:
        median_hz = get_speaker_median_f0(str(row['Speaker']))
        if median_hz and median_hz > 0:
            with np.errstate(divide='ignore', invalid='ignore'):
                pitch_plot = np.where(
                    np.isfinite(pitch_values),
                    1200.0 * np.log2(pitch_values / median_hz),
                    np.nan
                )
            ax_pitch.plot(pitch_times, pitch_plot, '-', linewidth=1.5, color='red')
            ax_pitch.set_ylabel('Pitch (cents, rel. speaker median)', color='red')
            ax_pitch.set_ylim(-1200, 1200)
            ax_pitch.axhline(0, color='red', linestyle='--', linewidth=0.6, alpha=0.5)
        else:
            # Fallback to Hz if median unavailable
            ax_pitch.plot(pitch_times, pitch_values, '-', linewidth=1.5, color='red')
            ax_pitch.set_ylabel('Pitch (Hz) [norm unavailable]', color='red')
            ax_pitch.set_ylim(50, 500)
    else:
        ax_pitch.plot(pitch_times, pitch_values, '-', linewidth=1.5, color='red')
        ax_pitch.set_ylabel('Pitch ($f_0$)', color='red')
        ax_pitch.set_ylim(50, 500)

    ax_pitch.tick_params(axis='y', colors='red')

    # Panel 2 – spectrogram (optional)
    if ax_spec is not None:
        librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=ax_spec)
        ax_spec.set(title='Spectrogram', ylabel='Hz')

    # Panel 3 – TextGrid (optional)
    if ax_tg is not None:
        tg_path = None
        # Look for the TextGrid in the exact same folder as the matched .wav file
        if audio_path:
            base_path = os.path.splitext(audio_path)[0]
            if os.path.exists(base_path + '.TextGrid'): tg_path = base_path + '.TextGrid'
            elif os.path.exists(base_path + '.textgrid'): tg_path = base_path + '.textgrid'

        if tg_path:
            try:
                tg = textgrid.TextGrid.fromFile(tg_path)
                n_tiers = len(tg.tiers)

                # Draw each tier from top to bottom
                for i, tier in enumerate(tg.tiers):
                    y_pos = n_tiers - i
                    ax_tg.axhline(y_pos - 0.5, color='black', lw=0.8) # Tier separator line

                    for interval in tier:
                        # Draw vertical boundary line
                        ax_tg.axvline(interval.maxTime, color='gray', linestyle=':', lw=1.5)
                        # Add the text annotation in the middle of the interval
                        if interval.mark.strip():
                            mid_time = (interval.minTime + interval.maxTime) / 2
                            ax_tg.text(mid_time, y_pos, interval.mark,
                                       ha='center', va='center', fontsize=11,
                                       bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=0))

                # Format the TextGrid axis
                ax_tg.set_ylim(0.5, n_tiers + 0.5)
                ax_tg.set_yticks(range(1, n_tiers + 1))
                ax_tg.set_yticklabels([tier.name for tier in reversed(tg.tiers)])
            except Exception as e:
                ax_tg.text(0.5, 0.5, f"Error loading TextGrid: {e}", ha='center', va='center', transform=ax_tg.transAxes)
        else:
            ax_tg.text(0.5, 0.5, "No TextGrid found", ha='center', va='center', color='gray', transform=ax_tg.transAxes)

        ax_tg.set_xlim(0, len(y)/sr) # Lock x-axis perfectly to the waveform above
        ax_tg.set_xlabel('Time (s)')
        ax_wave.set_xlabel('')

    return y, sr, file_name

In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# DEBOUNCING FOR BUTTON CLICKS
# ══════════════════════════════════════════════════════════════════════════════
# Prevents double-click lag (when user accidentally clicks button twice quickly)

import time
_last_click = 0
def _debounce():
    global _last_click
    if time.time() - _last_click < 0.2: return True
    _last_click = time.time()
    return False

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# SINGLE-COLUMN PLOT (Plot Audio button)
# ══════════════════════════════════════════════════════════════════════════════

def plot_audio_files(_):
    if _debounce(): return
    # Clear compare panes so only single-column view is visible
    output_left.clear_output(wait=False)
    output_left.outputs = ()
    output_right.clear_output(wait=False)
    output_right.outputs = ()
    output_box.clear_output(wait=False)
    output_box.outputs = ()
    with output_box:
        
        # Apply filters to get the subset of files to display
        filtered_df = apply_filters(
            tone_dropdown.value, vowel_length_dropdown.value,
            coda_consonant_dropdown.value, onset_dropdown.value,
            exclude_dropdown.value, dict_match_dropdown.value,
            dialect_dropdown.value, speaker_dropdown.value,
            word_id_dropdown.value, ons_type_dropdown.value
        )
        n          = n_input.value
        show_spec  = spectrogram_dropdown.value
        use_random = random_checkbox.value
        
        # If there are more matching files than n, we either take the first n or a random sample of n, based on the checkbox
        if use_random:
            display_df = filtered_df.sample(min(n, len(filtered_df)), random_state=None)
            print(f"Found {len(filtered_df)} matching files. Displaying {len(display_df)} randomly selected...")
        else:
            display_df = filtered_df.head(n)
            print(f"Found {len(filtered_df)} matching files. Displaying the first {n}...")

        print("Please wait, generating acoustic data...\n")
        
        # Loop through the selected files and render each one
        for _, row in display_df.iterrows():
            show_spectrogram = (show_spec == 'y')
            show_textgrid = show_tg_checkbox.value

            # Depending on the options, we create a different number of subplots with different height ratios to accommodate the waveform, spectrogram, and TextGrid tiers
            if show_spectrogram and show_textgrid:
                _, ax = plt.subplots(nrows=3, ncols=1, figsize=(10, 8), sharex=True, gridspec_kw={'height_ratios': [2, 2, 1]})
                ax_wave, ax_spec, ax_tg = ax[0], ax[1], ax[2]
            elif show_spectrogram and not show_textgrid:
                _, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 6), sharex=True, gridspec_kw={'height_ratios': [2, 2]})
                ax_wave, ax_spec, ax_tg = ax[0], ax[1], None
            elif not show_spectrogram and show_textgrid:
                _, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 5), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
                ax_wave, ax_spec, ax_tg = ax[0], None, ax[1]
            else:
                _, ax = plt.subplots(nrows=1, ncols=1, figsize=(10, 3))
                ax_wave, ax_spec, ax_tg = ax, None, None

            # Render the file and get the audio data for playback
            y, sr, _ = render_file(row, show_spectrogram, ax_wave, ax_spec, ax_tg, normalize_f0=normalize_f0_checkbox.value)
            plt.tight_layout()
            plt.show()
            # If audio data was successfully loaded, display the audio player
            if y is not None:
                display(Audio(data=y, rate=sr))

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# SIDE-BY-SIDE COMPARISON (Compare button)
# ══════════════════════════════════════════════════════════════════════════════

def plot_compare(_):
    if _debounce(): return

    # Clear all previous output panes before rendering compare results
    output_box.clear_output(wait=False)
    output_box.outputs = ()
    output_left.clear_output(wait=False)
    output_left.outputs = ()
    output_right.clear_output(wait=False)
    output_right.outputs = ()

    # Apply filters to get the subsets of files for left and right columns
    n          = n_input.value
    show_spec  = spectrogram_dropdown.value
    use_random = random_checkbox.value

    # Apply filters separately for left and right columns based on their respective dropdowns
    left_df  = apply_filters(
        tone_dropdown.value, vowel_length_dropdown.value,
        coda_consonant_dropdown.value, onset_dropdown.value,
        exclude_dropdown.value, dict_match_dropdown.value,
        dialect_dropdown.value, speaker_dropdown.value,
        word_id_dropdown.value, ons_type_dropdown.value
    )

    right_df = apply_filters(
        tone_R.value, vowel_length_R.value,
        coda_R.value, onset_R.value,
        exclude_R.value, dict_match_R.value,
        dialect_R.value, speaker_R.value,
        word_id_R.value, ons_type_R.value
    )

    # If there are more matching files than n in either column, we either take the first n or a random sample of n for that column, based on the checkbox
    if use_random:
        left_disp  = left_df.sample(min(n, len(left_df)),   random_state=None)
        right_disp = right_df.sample(min(n, len(right_df)), random_state=None)
    else:
        left_disp  = left_df.head(n)
        right_disp = right_df.head(n)

    # Display counts of total matching files and how many are being shown in each column
    with output_left:
        print(f"LEFT: {len(left_df)} total ({len(left_disp)} shown)")

    with output_right:
        print(f"RIGHT: {len(right_df)} total ({len(right_disp)} shown)")

    # Define a helper function to render a column of files given a filtered dataframe and an output widget to render into
    def _render_col(output_widget, disp_df, _):
        # Depending on the options, we create a different number of subplots with different height ratios to accommodate the waveform, spectrogram, and TextGrid tiers for each file in the column
        with output_widget:
            for _, row in disp_df.iterrows():
                show_spectrogram = (show_spec == 'y')
                show_textgrid = show_tg_checkbox.value

                if show_spectrogram and show_textgrid:
                    _, ax = plt.subplots(nrows=3, ncols=1, figsize=(6, 7), sharex=True, gridspec_kw={'height_ratios': [2, 2, 1]})
                    ax_wave, ax_spec, ax_tg = ax[0], ax[1], ax[2]
                elif show_spectrogram and not show_textgrid:
                    _, ax = plt.subplots(nrows=2, ncols=1, figsize=(6, 5), sharex=True, gridspec_kw={'height_ratios': [2, 2]})
                    ax_wave, ax_spec, ax_tg = ax[0], ax[1], None
                elif not show_spectrogram and show_textgrid:
                    _, ax = plt.subplots(nrows=2, ncols=1, figsize=(6, 4), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
                    ax_wave, ax_spec, ax_tg = ax[0], None, ax[1]
                else:
                    _, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 2.5))
                    ax_wave, ax_spec, ax_tg = ax, None, None

                # Render the file and get the audio data for playback
                y, sr, _ = render_file(row, show_spectrogram, ax_wave, ax_spec, ax_tg, normalize_f0=normalize_f0_checkbox.value)
                plt.tight_layout()
                plt.show()
                if y is not None:
                    display(Audio(data=y, rate=sr))

    _render_col(output_left,  left_disp,  "LEFT")
    _render_col(output_right, right_disp, "RIGHT")

In [16]:
# ══════════════════════════════════════════════════════════════════════════════
# WIRE UP BUTTONS AND BUILD UI LAYOUT
# ══════════════════════════════════════════════════════════════════════════════

# Clear previous button clicks using the public ipywidgets v8 API
search_button.on_click(plot_audio_files, remove=True)
compare_button.on_click(plot_compare, remove=True)

# Wire up buttons
search_button.on_click(plot_audio_files)
compare_button.on_click(plot_compare)

# Build UI layout
_h = lambda text: widgets.HTML(f"<b>{text}</b>")

# Clear the screen of any previously drawn UI elements before drawing new ones
clear_output(wait=True)
ui = widgets.VBox([
    _h("Left Column Filters"),
    widgets.HBox([tone_dropdown, vowel_length_dropdown, coda_consonant_dropdown,
                  onset_dropdown, exclude_dropdown]),
    widgets.HBox([dict_match_dropdown, dialect_dropdown, speaker_dropdown,
                  word_id_dropdown, ons_type_dropdown]),
    _h("Right Column Filters (Compare only)"),
    widgets.HBox([tone_R, vowel_length_R, coda_R, onset_R, exclude_R]),
    widgets.HBox([dict_match_R, dialect_R, speaker_R, word_id_R, ons_type_R]),
    _h("Options"),
    widgets.HBox([n_input, spectrogram_dropdown, show_tg_checkbox, random_checkbox, normalize_f0_checkbox]),
    widgets.HBox([search_button, compare_button]),
])

display(ui)
display(output_box)                                  # single-column output
display(widgets.HBox([output_left, output_right]))

Output()